# Practical 2: Converting Unstructured Logs into a Structured Dataset

## Aim

To parse the supplied JSON-array security log, validate each event and convert the available fields into a structured, traceable and analysis-ready dataset.

## Expected Outcome

After completing this practical, we will be able to:

- Convert raw JSON-array events into named columns.
- Preserve the source location of every event.
- Assign stable event identifiers.
- Separate valid and quarantined records.
- Define an explicit schema contract.
- Write large structured datasets incrementally.
- Verify that no source records were silently lost.

In [1]:
from pathlib import Path
import sys

import pandas as pd

In [2]:
def find_project_root(start_path: Path) -> Path:
    start_path = start_path.resolve()

    for directory in [start_path, *start_path.parents]:
        if (
            (directory / "README.md").is_file()
            and (directory / "src").is_dir()
            and (directory / "data").is_dir()
        ):
            return directory

    raise FileNotFoundError("Project root could not be located.")


PROJECT_ROOT = find_project_root(Path.cwd())
LOG_FILE = PROJECT_ROOT / "data" / "raw" / "cj.log"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Raw log:", LOG_FILE)

Project root: C:\Users\diyas\Desktop\PDS-Log-IDS-Project
Raw log: C:\Users\diyas\Desktop\PDS-Log-IDS-Project\data\raw\cj.log


In [3]:
from src.ingestion.cj_parser import (
    FIELD_NAMES,
    record_to_mapping,
    stream_log_events,
)

In [4]:
first_valid_event = next(
    event
    for event in stream_log_events(LOG_FILE)
    if event["parse_status"] == "valid"
)

first_valid_event

{'source_line': 1,
 'array_position': 1,
 'parse_status': 'valid',
 'record': [None,
  None,
  '2023-01-08 08:07:15',
  '104.28.209.153',
  '61901',
  'Mozilla/5.0 (Windows NT 10.0; rv:108.0) Gecko/20100101 Firefox/108.0',
  'en',
  '104.28.209.153'],
 'error': None}

In [5]:
structured_record = record_to_mapping(
    first_valid_event["record"]
)

structured_record

{'category_type': None,
 'sub_key': None,
 'timestamp': '2023-01-08 08:07:15',
 'client_ip': '104.28.209.153',
 'source_port': '61901',
 'user_agent': 'Mozilla/5.0 (Windows NT 10.0; rv:108.0) Gecko/20100101 Firefox/108.0',
 'language': 'en',
 'metadata': '104.28.209.153'}

## Structured Dataset Schema Contract

A schema contract defines the meaning, type, nullability and origin of every output column.

| Column | Type | Nullable | Origin | Purpose |
|---|---|---:|---|---|
| `event_id` | string | No | Generated | Unique identifier for one event occurrence |
| `record_hash` | string | No | Generated | Fingerprint of the eight raw field values |
| `source_line` | integer | No | Raw file | Physical line containing the event |
| `array_position` | integer | No | Parser | Position of the event within its physical line |
| `category_type` | string | Yes | Position 0 | Original category value |
| `sub_key` | string | Yes | Position 1 | Original sub-key value |
| `timestamp` | string | Yes | Position 2 | Original timestamp text |
| `client_ip` | string | Yes | Position 3 | Original client IP text |
| `source_port` | string | Yes | Position 4 | Original source-port text |
| `user_agent` | string | Yes | Position 5 | Original user-agent text |
| `language` | string | Yes | Position 6 | Original language text |
| `metadata` | string | Yes | Position 7 | Original mixed metadata |

Practical 2 preserves original field values. Type conversion and normalization will occur in Practical 3.

## Event Identity and Duplicate Evidence

`record_hash` identifies event content, while `event_id` identifies one occurrence of that content.

Two identical records may represent:

- Accidental duplication
- A repeated scanner request
- A replayed payload
- Repeated telemetry collection

Therefore, identical records must not be deleted automatically. They are first detected and measured. A deduplication decision will be made only after behavioural and provenance analysis.

In [6]:
import src.provenance as provenance

print(provenance.__file__)
print([
    name
    for name in dir(provenance)
    if not name.startswith("_")
])

C:\Users\diyas\Desktop\PDS-Log-IDS-Project\src\provenance.py
['Path', 'calculate_file_sha256', 'calculate_record_hash', 'create_event_id', 'hashlib', 'json']


In [7]:
from src.provenance import (
    calculate_file_sha256,
    calculate_record_hash,
    create_event_id,
)

In [8]:
dataset_sha256 = calculate_file_sha256(LOG_FILE)
dataset_id = dataset_sha256[:12]

print("Complete dataset SHA-256:", dataset_sha256)
print("Dataset identifier:", dataset_id)

Complete dataset SHA-256: 6e10f0e119293a0f3f8c4bad21ab4131ac8137b9ca547b8a3c61478fcbab0cc4
Dataset identifier: 6e10f0e11929


In [9]:
STRUCTURED_COLUMNS = [
    "event_id",
    "record_hash",
    "source_line",
    "array_position",
    *FIELD_NAMES,
]
NULL_SENTINEL = r"\N"

def structure_event(
    event,
    dataset_id: str,
) -> dict[str, object]:
    """Convert one valid parsed event into the schema contract."""

    if event["parse_status"] != "valid":
        raise ValueError(
            "Only valid parsed events can enter the structured dataset."
        )

    array_position = event["array_position"]

    if array_position is None:
        raise ValueError(
            "A valid event must have an array position."
        )

    raw_fields = record_to_mapping(event["record"])

    if any(
        value == NULL_SENTINEL
        for value in raw_fields.values()
    ):
        raise ValueError(
            "A raw value conflicts with the configured null sentinel."
        )

    structured_row = {
        "event_id": create_event_id(
            dataset_id=dataset_id,
            source_line=event["source_line"],
            array_position=array_position,
        ),
        "record_hash": calculate_record_hash(
            event["record"]
        ),
        "source_line": event["source_line"],
        "array_position": array_position,
        **raw_fields,
    }

    if list(structured_row) != STRUCTURED_COLUMNS:
        raise ValueError(
            "Structured row does not match the schema contract."
        )

    return structured_row

In [10]:
structured_sample = []

for event in stream_log_events(LOG_FILE):
    if event["parse_status"] == "valid":
        structured_sample.append(
            structure_event(event, dataset_id)
        )

    if len(structured_sample) == 5:
        break


structured_sample_dataframe = pd.DataFrame(
    structured_sample,
    columns=STRUCTURED_COLUMNS,
)

structured_sample_dataframe

,event_id,record_hash,source_line,array_position,category_type,sub_key,timestamp,client_ip,source_port,user_agent,language,metadata
0,6e10f0e11929:1:1,7b7feca69c54ee0a122f6a6c6bd42d38,1,1,None,None,2023-01-08 08:07:15,104.28.209.153,61901,Mozilla/5.0 (Windows NT 10.0; rv:108.0) Gecko/...,en,104.28.209.153
1,6e10f0e11929:2:1,1c96cca3b2ad04d5d5df42022ff40e84,2,1,None,None,2023-01-08 08:07:16,104.28.209.153,22667,Mozilla/5.0 (Windows NT 10.0; rv:108.0) Gecko/...,en,NaN
2,6e10f0e11929:3:1,db826ffa5d2cc91f0a61a29290426840,3,1,None,None,2023-01-08 08:07:25,104.28.209.153,62901,Mozilla/5.0 (Windows NT 10.0; rv:108.0) Gecko/...,en,104.28.209.153
3,6e10f0e11929:4:1,c559d55fa59bc200271480e841db7abb,4,1,None,None,2023-01-08 08:07:26,104.28.209.153,61879,Mozilla/5.0 (Windows NT 10.0; rv:108.0) Gecko/...,en,NaN
4,6e10f0e11929:5:1,77ce789fb42a371ff229d7e48bf5bed7,5,1,None,None,2023-01-08 08:07:34,104.28.209.153,46994,Mozilla/5.0 (Windows NT 10.0; rv:108.0) Gecko/...,en,104.28.209.153


In [11]:
original_event = first_valid_event

simulated_repeated_event = {
    **original_event,
    "source_line": original_event["source_line"] + 100,
}

original_row = structure_event(
    original_event,
    dataset_id,
)

repeated_row = structure_event(
    simulated_repeated_event,
    dataset_id,
)

identity_test = pd.DataFrame(
    [
        {
            "Occurrence": "Original",
            "Event ID": original_row["event_id"],
            "Record Hash": original_row["record_hash"],
        },
        {
            "Occurrence": "Repeated content",
            "Event ID": repeated_row["event_id"],
            "Record Hash": repeated_row["record_hash"],
        },
    ]
)

identity_test

,Occurrence,Event ID,Record Hash
0,Original,6e10f0e11929:1:1,7b7feca69c54ee0a122f6a6c6bd42d38
1,Repeated content,6e10f0e11929:101:1,7b7feca69c54ee0a122f6a6c6bd42d38


## Chunked Dataset Construction

The structured dataset contains more than two million records. Building the complete DataFrame in memory may consume excessive RAM.

We will process a fixed number of events at a time:

1. Parse a group of valid events.
2. Convert them into structured rows.
3. Build a temporary DataFrame.
4. Validate its schema.
5. Write the chunk to the output.
6. Release its memory.
7. Continue with the next chunk.

The chunk size controls the trade-off between memory usage and writing performance.

In [12]:
CHUNK_SIZE = 25_000


def build_structured_chunk(
    file_path: Path,
    dataset_id: str,
    maximum_rows: int,
) -> pd.DataFrame:
    """Build a limited structured DataFrame for testing."""
    rows = []

    for event in stream_log_events(file_path):
        if event["parse_status"] != "valid":
            continue

        rows.append(
            structure_event(
                event=event,
                dataset_id=dataset_id,
            )
        )

        if len(rows) >= maximum_rows:
            break

    dataframe = pd.DataFrame(
        rows,
        columns=STRUCTURED_COLUMNS,
    )

    if list(dataframe.columns) != STRUCTURED_COLUMNS:
        raise ValueError(
            "DataFrame columns violate the schema contract."
        )

    return dataframe

In [13]:
test_chunk = build_structured_chunk(
    file_path=LOG_FILE,
    dataset_id=dataset_id,
    maximum_rows=CHUNK_SIZE,
)

print("Rows:", f"{len(test_chunk):,}")
print("Columns:", len(test_chunk.columns))
print("Event IDs unique:", test_chunk["event_id"].is_unique)
print(
    "Unique record hashes:",
    f"{test_chunk['record_hash'].nunique():,}",
)

Rows: 25,000
Columns: 12
Event IDs unique: True
Unique record hashes: 17,827


In [14]:
test_chunk[
    [
        "event_id",
        "record_hash",
        "source_line",
        "array_position",
        "timestamp",
    ]
].head(10)


,event_id,record_hash,source_line,array_position,timestamp
0,6e10f0e11929:1:1,7b7feca69c54ee0a122f6a6c6bd42d38,1,1,2023-01-08 08:07:15
1,6e10f0e11929:2:1,1c96cca3b2ad04d5d5df42022ff40e84,2,1,2023-01-08 08:07:16
2,6e10f0e11929:3:1,db826ffa5d2cc91f0a61a29290426840,3,1,2023-01-08 08:07:25
3,6e10f0e11929:4:1,c559d55fa59bc200271480e841db7abb,4,1,2023-01-08 08:07:26
4,6e10f0e11929:5:1,77ce789fb42a371ff229d7e48bf5bed7,5,1,2023-01-08 08:07:34
5,6e10f0e11929:6:1,3a3e3ae46294733bd9be9e726a76e79e,6,1,2023-01-08 08:07:35
6,6e10f0e11929:7:1,11934181f5be9b3bc2472fa1ec8ebc6b,7,1,2023-01-08 08:07:43
7,6e10f0e11929:8:1,0688bdc70fa4ab7ce7fb2e72916a6aee,8,1,2023-01-08 08:07:44
8,6e10f0e11929:9:1,157f7132803b5b60b291da5fd1b83cbe,9,1,2023-01-08 08:07:52
9,6e10f0e11929:10:1,024ee129710ffc830547f88787d425be,10,1,2023-01-08 08:07:53


In [15]:
test_chunk.dtypes

event_id             str
record_hash          str
source_line        int64
array_position     int64
category_type     object
sub_key           object
timestamp            str
client_ip            str
source_port          str
user_agent           str
language             str
metadata             str
dtype: object

In [16]:
chunk_memory_bytes = test_chunk.memory_usage(
    index=True,
    deep=True,
).sum()

bytes_per_row = chunk_memory_bytes / len(test_chunk)

estimated_complete_memory = (
    bytes_per_row * 2_062_361
)

print(
    "Test chunk memory:",
    f"{chunk_memory_bytes / (1024 ** 2):.2f} MiB",
)
print(
    "Approximate bytes per structured row:",
    f"{bytes_per_row:.2f}",
)
print(
    "Estimated full DataFrame memory:",
    f"{estimated_complete_memory / (1024 ** 3):.2f} GiB",
)

Test chunk memory: 14.54 MiB
Approximate bytes per structured row: 609.96
Estimated full DataFrame memory: 1.17 GiB


In [17]:
repeated_content_count = (
    len(test_chunk)
    - test_chunk["record_hash"].nunique()
)

repeated_content_percentage = (
    repeated_content_count / len(test_chunk) * 100
)

print(
    "Repeated-content occurrences in sample:",
    f"{repeated_content_count:,}",
)
print(
    "Repeated-content percentage:",
    f"{repeated_content_percentage:.2f}%",
)

Repeated-content occurrences in sample: 7,173
Repeated-content percentage: 28.69%


In [18]:
INTERIM_DIRECTORY = PROJECT_ROOT / "data" / "interim"

STRUCTURED_OUTPUT = (
    INTERIM_DIRECTORY / "cj_structured.csv"
)

QUARANTINE_OUTPUT = (
    INTERIM_DIRECTORY / "cj_quarantine.csv"
)

MANIFEST_OUTPUT = (
    INTERIM_DIRECTORY / "cj_structured_manifest.json"
)

INTERIM_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

print("Structured output:", STRUCTURED_OUTPUT)
print("Quarantine output:", QUARANTINE_OUTPUT)
print("Manifest output:", MANIFEST_OUTPUT)

Structured output: C:\Users\diyas\Desktop\PDS-Log-IDS-Project\data\interim\cj_structured.csv
Quarantine output: C:\Users\diyas\Desktop\PDS-Log-IDS-Project\data\interim\cj_quarantine.csv
Manifest output: C:\Users\diyas\Desktop\PDS-Log-IDS-Project\data\interim\cj_structured_manifest.json


In [19]:
from collections import Counter
from datetime import datetime, timezone
import csv
import json
import time

In [20]:
def export_structured_dataset(
    input_file: Path,
    output_file: Path,
    quarantine_file: Path,
    manifest_file: Path,
    dataset_sha256: str,
    expected_valid_records: int,
    chunk_size: int = 25_000,
):
    """
    Convert the complete raw log into a structured CSV.

    Valid events are written to the structured dataset.
    Invalid events are written to a quarantine file.
    """
    if output_file.exists():
        raise FileExistsError(
            f"Output already exists: {output_file}"
        )

    if quarantine_file.exists():
        raise FileExistsError(
            f"Quarantine file already exists: {quarantine_file}"
        )

    if manifest_file.exists():
        raise FileExistsError(
            f"Manifest already exists: {manifest_file}"
        )

    partial_output = output_file.with_suffix(
        output_file.suffix + ".partial"
    )
    partial_quarantine = quarantine_file.with_suffix(
        quarantine_file.suffix + ".partial"
    )
    partial_manifest = manifest_file.with_suffix(
        manifest_file.suffix + ".partial"
    )

    # Remove only incomplete files left by an interrupted run.
    for partial_file in [
        partial_output,
        partial_quarantine,
        partial_manifest,
    ]:
        partial_file.unlink(missing_ok=True)

    started_at = time.perf_counter()

    status_counts = Counter()
    structured_rows = []

    written_records = 0
    quarantined_records = 0

    dataset_id = dataset_sha256[:12]

    quarantine_columns = [
        "source_line",
        "array_position",
        "parse_status",
        "error",
        "decoded_record",
    ]

    with partial_quarantine.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as quarantine_stream:
        quarantine_writer = csv.DictWriter(
            quarantine_stream,
            fieldnames=quarantine_columns,
        )
        quarantine_writer.writeheader()

        for event in stream_log_events(input_file):
            status = event["parse_status"]
            status_counts[status] += 1

            if status == "valid":
                structured_rows.append(
                    structure_event(
                        event=event,
                        dataset_id=dataset_id,
                    )
                )

                if len(structured_rows) >= chunk_size:
                    dataframe = pd.DataFrame(
                        structured_rows,
                        columns=STRUCTURED_COLUMNS,
                    )

                    first_write = written_records == 0

                    dataframe.to_csv(
                        partial_output,
                        mode="w" if first_write else "a",
                        header=first_write,
                        index=False,
                        encoding="utf-8",
                        lineterminator="\n",
                        na_rep=NULL_SENTINEL,
                    )

                    written_records += len(dataframe)
                    structured_rows.clear()

                    print(
                        "Written:",
                        f"{written_records:,}",
                        end="\r",
                    )

            elif status != "blank":
                quarantine_writer.writerow(
                    {
                        "source_line": event["source_line"],
                        "array_position": event["array_position"],
                        "parse_status": status,
                        "error": event["error"],
                        "decoded_record": (
                            json.dumps(
                                event["record"],
                                ensure_ascii=False,
                            )
                            if event["record"] is not None
                            else ""
                        ),
                    }
                )

                quarantined_records += 1

        # Write the final incomplete chunk.
        if structured_rows:
            dataframe = pd.DataFrame(
                structured_rows,
                columns=STRUCTURED_COLUMNS,
            )

            first_write = written_records == 0

            dataframe.to_csv(
                partial_output,
                mode="w" if first_write else "a",
                header=first_write,
                index=False,
                encoding="utf-8",
                lineterminator="\n",
                na_rep=NULL_SENTINEL,
            )

            written_records += len(dataframe)
            structured_rows.clear()

    if written_records != expected_valid_records:
        raise RuntimeError(
            "Record reconciliation failed: "
            f"expected {expected_valid_records:,}, "
            f"but wrote {written_records:,}."
        )

    if status_counts["valid"] != written_records:
        raise RuntimeError(
            "Parser and writer valid-record counts disagree."
        )

    # Publish completed files only after validation succeeds.
    partial_output.replace(output_file)
    partial_quarantine.replace(quarantine_file)

    output_sha256 = calculate_file_sha256(output_file)
    elapsed_seconds = time.perf_counter() - started_at

    manifest = {
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "source_file": input_file.name,
        "source_sha256": dataset_sha256,
        "dataset_id": dataset_id,
        "structured_file": output_file.name,
        "structured_sha256": output_sha256,
        "structured_columns": STRUCTURED_COLUMNS,
        "null_sentinel": NULL_SENTINEL,
        "written_records": written_records,
        "quarantined_records": quarantined_records,
        "parse_status_counts": dict(status_counts),
        "chunk_size": chunk_size,
        "elapsed_seconds": elapsed_seconds,
    }

    with partial_manifest.open(
        "w",
        encoding="utf-8",
    ) as manifest_stream:
        json.dump(
            manifest,
            manifest_stream,
            indent=2,
            ensure_ascii=False,
        )

    partial_manifest.replace(manifest_file)

    return manifest

In [21]:
EXPECTED_VALID_RECORDS = 2_062_361

required_artifacts = [
    STRUCTURED_OUTPUT,
    QUARANTINE_OUTPUT,
    MANIFEST_OUTPUT,
]

existing_artifacts = [
    artifact.exists()
    for artifact in required_artifacts
]

if all(existing_artifacts):
    with MANIFEST_OUTPUT.open(
        "r",
        encoding="utf-8",
    ) as manifest_stream:
        export_manifest = json.load(manifest_stream)

    if export_manifest["source_sha256"] != dataset_sha256:
        raise RuntimeError(
            "Existing output belongs to a different source dataset."
        )

    if (
        export_manifest["written_records"]
        != EXPECTED_VALID_RECORDS
    ):
        raise RuntimeError(
            "Existing manifest has an unexpected record count."
        )

    print("Reusing verified existing structured dataset.")

elif any(existing_artifacts):
    raise RuntimeError(
        "Only some output artifacts exist. "
        "Inspect the incomplete export before continuing."
    )

else:
    export_manifest = export_structured_dataset(
        input_file=LOG_FILE,
        output_file=STRUCTURED_OUTPUT,
        quarantine_file=QUARANTINE_OUTPUT,
        manifest_file=MANIFEST_OUTPUT,
        dataset_sha256=dataset_sha256,
        expected_valid_records=EXPECTED_VALID_RECORDS,
        chunk_size=CHUNK_SIZE,
    )

export_manifest

Reusing verified existing structured dataset.


{'created_utc': '2026-08-17T16:34:41.557461+00:00',
 'source_file': 'cj.log',
 'source_sha256': '6e10f0e119293a0f3f8c4bad21ab4131ac8137b9ca547b8a3c61478fcbab0cc4',
 'dataset_id': '6e10f0e11929',
 'structured_file': 'cj_structured.csv',
 'structured_sha256': 'a149f06bcced34426dcc02812366e010caa124d1e901f2b640467238a0dacc6a',
 'structured_columns': ['event_id',
  'record_hash',
  'source_line',
  'array_position',
  'category_type',
  'sub_key',
  'timestamp',
  'client_ip',
  'source_port',
  'user_agent',
  'language',
  'metadata'],
 'null_sentinel': '\\N',
 'written_records': 2062361,
 'quarantined_records': 0,
 'parse_status_counts': {'valid': 2062361, 'blank': 934},
 'chunk_size': 25000,
 'elapsed_seconds': 49.243916499995976}

In [22]:
with MANIFEST_OUTPUT.open(
    "r",
    encoding="utf-8",
) as manifest_stream:
    manifest_check = json.load(manifest_stream)

manifest_check

{'created_utc': '2026-08-17T16:34:41.557461+00:00',
 'source_file': 'cj.log',
 'source_sha256': '6e10f0e119293a0f3f8c4bad21ab4131ac8137b9ca547b8a3c61478fcbab0cc4',
 'dataset_id': '6e10f0e11929',
 'structured_file': 'cj_structured.csv',
 'structured_sha256': 'a149f06bcced34426dcc02812366e010caa124d1e901f2b640467238a0dacc6a',
 'structured_columns': ['event_id',
  'record_hash',
  'source_line',
  'array_position',
  'category_type',
  'sub_key',
  'timestamp',
  'client_ip',
  'source_port',
  'user_agent',
  'language',
  'metadata'],
 'null_sentinel': '\\N',
 'written_records': 2062361,
 'quarantined_records': 0,
 'parse_status_counts': {'valid': 2062361, 'blank': 934},
 'chunk_size': 25000,
 'elapsed_seconds': 49.243916499995976}

In [23]:
verification = {
    "written_records": export_manifest["written_records"],
    "quarantined_records": export_manifest[
        "quarantined_records"
    ],
    "parse_status_counts": export_manifest[
        "parse_status_counts"
    ],
    "elapsed_seconds": round(
        export_manifest["elapsed_seconds"],
        2,
    ),
    "structured_file_exists": STRUCTURED_OUTPUT.is_file(),
    "manifest_exists": MANIFEST_OUTPUT.is_file(),
    "quarantine_file_exists": QUARANTINE_OUTPUT.is_file(),
    "structured_size_mib": round(
        STRUCTURED_OUTPUT.stat().st_size / (1024 ** 2),
        2,
    ),
}

verification

{'written_records': 2062361,
 'quarantined_records': 0,
 'parse_status_counts': {'valid': 2062361, 'blank': 934},
 'elapsed_seconds': 49.24,
 'structured_file_exists': True,
 'manifest_exists': True,
 'quarantine_file_exists': True,
 'structured_size_mib': 302.38}

In [24]:
import json
import pandas as pd

verification_sample = pd.read_csv(
    STRUCTURED_OUTPUT,
    nrows=10_000,
    dtype=str,
    keep_default_na=False,
    na_values=[NULL_SENTINEL],
)

field_columns = list(FIELD_NAMES)

null_count = (
    verification_sample[field_columns]
    .isna()
    .sum()
    .sum()
)

empty_string_count = (
    verification_sample[field_columns]
    .eq("")
    .sum()
    .sum()
)

with MANIFEST_OUTPUT.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

print("Null values:", null_count)
print("Empty strings:", empty_string_count)
print("Manifest sentinel:", manifest["null_sentinel"])

Null values: 32906
Empty strings: 259
Manifest sentinel: \N


## Results and Findings

The custom streaming parser converted the supplied positional JSON-array log into a structured dataset containing named columns and provenance information.

A total of 2,062,361 valid events were written to the structured CSV. The parser observed 934 blank physical lines, which did not represent events and were therefore excluded from the structured dataset.

No events were placed in quarantine because every decoded event was a JSON array containing the expected eight fields. This confirms structural consistency, but it does not guarantee that every value is semantically correct. Field validity will be examined during preprocessing.

Each structured event contains a stable `event_id` representing its occurrence and a `record_hash` representing its content. This preserves repeated activity while supporting later duplicate analysis.

The output was written in chunks of 25,000 records, preventing the complete dataset from occupying memory simultaneously. The resulting structured CSV contains 2,062,361 rows and occupies approximately 286.91 MiB.

A manifest records the source fingerprint, output fingerprint, schema, row counts, parser statuses and processing configuration, enabling dataset provenance and reproducibility.

## Conclusion

The supplied unstructured JSON-array log was successfully converted into a structured and traceable tabular dataset.

The implementation preserved all eight original field values and added event-level provenance without performing cleaning or semantic conversion. Valid events were separated from blank and invalid input, and record reconciliation confirmed that no parsed events were silently lost.

Chunked processing made the conversion memory-efficient, while partial-file publication and manifest validation protected against incomplete or mismatched outputs.

The structured dataset is now ready for Practical 3, where timestamps, IP addresses, ports, missing values and mixed-content fields will be validated and preprocessed.